In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, StringType
from pyspark.sql import functions as F

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("HomeCredit_Serving_Layer") \
    .enableHiveSupport() \
    .getOrCreate()
raw_app = spark.read.parquet("/user/student/home_credit/staging/application_train")
raw_prev = spark.read.parquet("/user/student/home_credit/staging/previous_application")
raw_inst = spark.read.parquet("/user/student/home_credit/staging/installments_payments")
raw_bureau = spark.read.parquet("/user/student/home_credit/staging/bureau")

2026-09-04 02:43:22,257 WARN util.Utils: Your hostname, localhost.localdomain resolves to a loopback address: 127.0.0.1, but we couldn't find any external IP address!
2026-09-04 02:43:22,293 WARN util.Utils: Set SPARK_LOCAL_IP if you need to bind to another address
2026-09-04 02:43:38,509 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
2026-09-04 02:44:02,666 WARN internal.MacAddressUtil: Failed to find a usable hardware address from the network interfaces; using random bytes: 9e:fd:04:4d:ce:11:84:13


In [13]:
from pyspark.sql import functions as F

# Repayment History Aggregation

repayment_features = (
    clean_inst
    
    # 1. Calculate Days Past Due for each payment
    .withColumn(
        "days_past_due",
        F.when(
            F.col("DAYS_ENTRY_PAYMENT").isNotNull(),
            F.greatest(
                F.col("DAYS_INSTALMENT") - F.col("DAYS_ENTRY_PAYMENT"),
                F.lit(0)
            )
        ).otherwise(F.lit(0))
    )


    # 2. Aggregate repayment history
    #    for each previous loan

    .groupBy("SK_ID_PREV")
    .agg(
        F.sum("days_past_due").alias("total_days_past_due"),

        F.sum(
            F.when(
                F.col("days_past_due") > 0,
                1
            ).otherwise(0)
        ).alias("num_late_payments")
    )
)

# ==========================================
# Check the result
# ==========================================

repayment_features.show(20, truncate=False)

+----------+-------------------+-----------------+
|SK_ID_PREV|total_days_past_due|num_late_payments|
+----------+-------------------+-----------------+
|1382535   |390.0              |9                |
|2726379   |1208.0             |90               |
|1651129   |176.0              |4                |
|1576426   |278.0              |18               |
|1597372   |170.0              |9                |
|2524431   |85.0               |6                |
|1568739   |862.0              |57               |
|2173518   |419.0              |28               |
|2233640   |346.0              |22               |
|1100657   |169.0              |10               |
|2409566   |173.0              |10               |
|1634555   |73.0               |4                |
|2528096   |27.0               |7                |
|1234770   |700.0              |40               |
|1882058   |314.0              |12               |
|1644512   |93.0               |10               |
|2463064   |40.0               